# 衔接与生成——从三算子对照到 gelu_approx 交付

> **衔接与生成**（建议 ~2h，含实操）。前置要求：基础课程 01.01~02.03（半天课/全天基础课/自学均可）。产物 ①：gelu_approx 工程 + 全量测试证据。

> **本节新增知识只有 2 件**：① **精度预算**：8 步链的误差累积意识（基础课程只到单步容差判据）；② **指挥官实操**：问卷确认、观察推进、判读自己的验收报告。其余全部是基础课程知识的延续：三层架构、验收对偶（一件事由两个独立角色"一个做、一个查"）、需求五要素、golden 容差（golden：双精度 CPU 算出的参考答案，测试拿它判卷）。只是这次，由你亲手触发。

> **写给第一次接触算子的读者**：算子代码由 AI 团队编写与修复，指挥官角色看得懂结果、做得了决定就够。本路径做出**四件产物**（四个可验收的成果）。

本环节不复讲概念，只**激活**已有知识，建立实战的进阶预期。先看一张贯穿本路径的对照表：

## 1. 三个算子，一条进阶线

在本路径中，你将与三个算子相遇，角色逐级升级：

| 维度 | add_custom<br>（基础课程·01.02） | xlog1py<br>（基础课程·02.01~02.02） | gelu_approx<br>（本路径·03.01~03.04） |
|---|---|---|---|
| 你的角色 | **跟着做**（模仿者） | **看着学**（观察者） | **指挥干**（指挥官） |
| 工作流 | Flash 版八阶段 | 完整版 v3.0.0（实录精读） | 完整版 v3.0.0（亲指挥） |
| 计算链 | 1 步加法 | 3 步（Ln+修正+Mul） | **8 步**（含 Tanh） |
| 精度 | 位级精确（17/17，max_diff=0） | 40/40 全绿 | 预算紧张（裕量 23%）→ 分层修复 |
| 性能话题 | 不涉及 | 访存受限成立（64% 峰值） | 假设需重验（48%） |
| 深度 | 会用 | 理解机制 | 精通：预算 / 分诊 / 密度 / 反哺 |

从模仿一次交付，到理解机制设计，再到独立指挥并反哺生态，现在进入指挥官角色。
<img src="./images/03.01_three_operators.png" alt="三个算子，一条进阶线：add_custom 跟着做 → xlog1py 看着学 → gelu 指挥干" width="700px">

> 基础课还亲手交付过 sigmoid_custom（01.02 练习，Flash 版第二算子），那是从跟着做到自己能做的检验；本路径则是指挥 AI 做。

**你的工具箱 → 本路径怎么用**：

| 基础课给你的工具 | 在本路径的使用场景 |
|---|---|
| 需求五要素（01.02） | 03.01 §6.1 读需求卡 |
| 完整版三思想（02.01） | 03.01 §5.4 机制速讲，换指挥官视角亲验 |
| 调试路由表（01.03） | 03.02 §2 分诊，从找哪个 Skill 升级到定位哪层根因 |
| 三原则（02.02） | 03.03 §2.1 回顾，换算子重验假设 |
| PR 八步与红线（02.03） | 03.04 §4 贡献路径，换目标仓原样复用 |

### 为什么是 gelu_approx？

工具箱已经就绪，但为什么目标是 gelu_approx，而不是别的算子？

不是因为它最重要，而是因为它的复杂度**刚好在每一层知识面上暴露出真问题**：

| 如果用… | 你会错过… |
|---|---|
| add_custom（1 步链） | 加法是精确运算（max_diff=0），精度、调试、性能全无题可出 |
| xlog1py（3 步链） | 基础课已用过；naive 实现就能全过，没有调试素材 |
| MatMul 等复杂算子 | 需要 Cube / Tiling 前置知识，超出本路径范围 |
| **gelu_approx（8 步链）** | **每层都有真问题**：精度预算只剩 23%、4 个真机失败用例、计算密度 4 倍 |

> **一句话**：add_custom 教你怎么开车，xlog1py 教你看导航，gelu 教你修车、调教、改装。它的 4 个失败不是虚构的，是 Ascend 910B3 真机跑出来的，每个失败精准对应一个知识层（算法层 / API 约束层 / 指令层）。

理解了目标，现在确认出发状态：


## 2. 你已经会什么（对照激活）

| 你学过的（基础课程出处） | 本路径的进阶方向 |
|---|---|
| 三层架构 Plugin/Agent/Skill（01.01） | 深入 harness（总调度器）内部机制 |
| Flash 八阶段 + add_custom 实操（01.02） | 完整版七节点 + **gelu_approx**（8 步计算链）实操 |
| 调试 Skill 路由表（01.03） | 三个真实坑的**根因级分析**与修复 |
| v3.0.0 三大思想：验收对偶/边界静默/过程产物分离（02.01） | 在实操中**亲验**每一条 |
| 性能三原则：数据先于观点/分档读法/流量思维（02.02） | 完整闭环实操 + **计算密度分析**（新） |
| GitCode PR 八步（02.03） | Skill 修改者视角的**贡献实操** |

**课前必做**：运行下面的环境自检，缺项按建议补齐或记下降级方案。


In [ ]:
# 开始前自检：逐项检查环境（缺失项打印降级建议，不报错）
import shutil, subprocess, sys

def check_env():
    """检查本路径所需环境，逐项输出 OK/缺失 + 建议"""
    checks = [
        ("Python ≥ 3.10", sys.version_info >= (3, 10), f"当前 {sys.version.split()[0]}"),
        ("Node.js ≥ 20.11 (harness 依赖)", None, ""),
        ("tmux (后台会话)", None, ""),
        ("torch_npu (NPU 接入)", None, ""),
        ("OpenCode CLI (Agent 宿主)", None, ""),
    ]
    for cmd in ["node --version", "tmux -V"]:
        try:
            r = subprocess.run(cmd.split(), capture_output=True, text=True, timeout=5)
            ver = r.stdout.strip() or r.stderr.strip()
        except (FileNotFoundError, subprocess.TimeoutExpired):
            ver = None
    try:
        import torch_npu; npu_ok = True; npu_info = f"torch_npu {torch_npu.__version__}"
    except ImportError:
        npu_ok = False; npu_info = "未安装"

    results = [
        ("Python ≥ 3.10", True, sys.version.split()[0]),
        ("Node.js ≥ 20.11", shutil.which("node") is not None, ""),
        ("tmux", shutil.which("tmux") is not None, ""),
        ("torch_npu", npu_ok, npu_info),
        ("OpenCode CLI", shutil.which("opencode") is not None, ""),
    ]
    print("环境自检（无 NPU 时本路径 Lab 需在 CANNLab 或本地昇腾环境完成）")
    print("═" * 55)
    for name, ok, info in results:
        mark = "✓" if ok else "✗"
        print(f"  [{mark}] {name:<28s} {info}")
    missing = [r[0] for r in results if not r[1]]
    if missing:
        print(f"\n缺 {len(missing)} 项：{', '.join(missing)}")
        print("→ 无 NPU 环境可在 CANNLab 云环境完成实操（免费，预装）")
    else:
        print("\n✓ 环境就绪，可进入全天实操")

check_env()

<details><summary>已知环境坑（课程开发实录踩坑）</summary>

版本锚定：`cann/cannbot` @ `7cf1845` · `cann/cannbot-skills` @ `94216ba2`（2026-09-30）

| 症状 | 修复 |
|---|---|
| install-helper 命令未找到 / 列表为空 / AI 工具不识别 Skills / 软链接失效 | `install-helper doctor --fix` 自动诊断修复；AI 工具重启或新开会话；核对 Node ≥ 18 |
| 执行者反复"研究很久却零产出"、空转重试 | 子会话读 `.cannbot/dependencies/` 触发 external_directory 权限询问，非交互模式被自动拒绝。目标工程 `.opencode/opencode.json` 加 `"permission": {"external_directory": "allow"}`（样例见 `src/opencode_permission_sample.json`） |
| 某节点会话以 reason=length 结束、无交付 | 执行者单步推理超长截断（已知行为）。**不干预**：verifier 打回 + harness 自动重试自愈；连续两次以上再人工介入 |
| 知识搜集节点重试耗尽、整轮退出 | 设计内行为（可裁剪可选节点）。PM 规划下一轮：省略知识搜集、复用已验收产物 |
| verifier 写了报告但节点反复重验 | verdict_missing（未提交公开裁决，平台已知行为）。观察即可；重试通常自愈 |
| 测试报告显示 op_available_at_run: false、全 skip | wheel 未安装或未 import 注册。`pip install --force-reinstall --no-deps dist/*.whl`；测试文件首行 import 必须生效 |
| 素材工程测试全绿（应为 4 败 10 过） | pip 环境装了修复版 wheel，覆盖了 naive 版。重装 naive：`cd src/gelu_approx_naive && bash build.sh --install` |
| 首次安装后依赖仓未就位 / cann_bench_utils 缺失 | 确认 `.cannbot/dependencies/ops-direct-invoke/` 下 4 个依赖仓就位；cann_bench_utils 按 cann-bench 的 `ensure_cann_bench_utils.sh` 编译安装 |

</details>

## 3. 三个当时没展开的问题

对照上面的激活表，基础课程已覆盖会用的完整能力。进入实战之前，先思考这三个问题：它们在基础课程里都没有展开，本路径会逐一回答（问题 1 本节 5 就答）。

**问题 1（精度）**：基础课程的 xlog1py（02.01~02.02）计算链只有 3 步，精度验收轻松通过。如果一个算子的计算链有 **8 步**，每一步的浮点舍入误差累积起来，还能通过 rtol=1e-5 / atol=1e-7 的红线吗？（7.5 回答）

**问题 2（调试）**：01.03（基础课程）的调试路由表能告诉你该加载哪个 Skill。但加载之后，根因可能藏在**三个不同的深度**：所写的公式本身、平台 API 的隐藏约束、硬件指令的精度规格。怎么区分、从哪里查起？（[03.02](./03.02_operator_debug.ipynb)回答）

**问题 3（性能）**：02.02 说 elementwise 算子是访存受限，优化方向是让搬运更满。但那个结论是对 3 步链的 xlog1py 说的。如果一个算子每搬运 1 字节要执行 **4 倍**的指令，访存受限还成立吗？（[03.03](./03.03_performance_engineering.ipynb)回答）

三个问题分别指向**精度、调试、性能**在多步链场景下的新形态，这就是会用与精通的差距。

> 💡 **先记下对三个问题的直觉答案**，学完本路径回来对照，看哪一个问题改变了判断。

问题 1 现在就能感受。运行下面的模拟：对比不同链长的误差累积，看看误差随步数怎么增长。


In [ ]:
# 链长 vs 精度累积：蒙特卡洛模拟舍入误差的增长规律（CPU，无 NPU）
import numpy as np

def simulate_chain_error(n_steps, n_samples=20000, seed=42):
    """模拟 n 步计算链的 fp32 舍入累积：每步加小扰动后舍入到 fp32（数值保持在 1 附近，ulp 恒定）"""
    rng = np.random.default_rng(seed)
    a32 = np.ones(n_samples, dtype=np.float32)   # fp32 链
    a64 = np.ones(n_samples)                     # fp64 无舍入参考
    for _ in range(n_steps):
        eps = rng.uniform(-0.01, 0.01, n_samples).astype(np.float32)
        a32 = (a32 + eps).astype(np.float32)     # 每步一次 fp32 舍入
        a64 = a64 + eps.astype(np.float64)
    d = np.abs(a32.astype(np.float64) - a64)
    return np.median(d), np.max(d)

print("计算链长度 vs 舍入误差累积（蒙特卡洛 20000 样本，数值保持在 1 附近）")
print("═" * 58)
print(f"{'链长':>4s}  {'中位误差':>10s}  {'最大误差':>10s}  {'√N×ulp/2':>10s}")
print("─" * 58)
ulp = 1.19e-7  # fp32 在 1 附近的 ulp
for n in [1, 3, 8, 16, 32]:
    med, mx = simulate_chain_error(n)
    print(f"{n:>4d}  {med:>10.2e}  {mx:>10.2e}  {np.sqrt(n)*ulp/2:>10.2e}")
print("─" * 58)
print("→ 误差随链长按 √N 增长（相邻档比值 ≈ √2），不是线性放大")
print("→ 8 步链误差 ≈ 3 步链的 1.6 倍（√8/√3）")
print("→ 真实 gelu 8 步链的预算有多紧？本节 5 用真实公式实测")


## 4. 衔接自查（5 题，全部答对再进入第 5 节）

> 用判分 cell 逐题确认；第 5 题为简答，对照折叠参考答案。

1. **（单选）** Flash 版和完整版的本质区别是？
   - A. Flash 用 Python，完整版用 C++
   - B. Flash 单 Agent 自走全流程，完整版 PM + 多角色协作 + 节点级独立验收
   - C. Flash 更快，完整版更慢，其他一样
   - D. Flash 不需要 NPU

2. **（单选）** v3.0.0 中某节点验收失败后，harness 会？
   - A. 暂停等用户修复
   - B. 自动重试该节点（默认最多 3 次），失败原因注入下一次执行
   - C. 跳过继续
   - D. 整个工作流作废

3. **（单选）** 性能三原则中分档读法指的是？
   - A. 按数据类型分档
   - B. L 档看带宽 vs HBM 峰值，S/M 档看时延恒定性
   - C. 按测试等级分档
   - D. 按 NPU 型号分档

4. **（单选）** `.cannbot/<任务名>/workflow<N>/` 存放的是什么？
   - A. 算子源代码
   - B. 过程产物（workflow YAML、验收报告、测试执行记录、状态）
   - C. 安装配置
   - D. 测试用例数据

5. **（简答）** gelu_approx 的计算链比 xlog1py 长几倍？这带来什么新挑战？

<details><summary>参考答案</summary>

1. **B** 2. **B** 3. **B** 4. **B**
5. xlog1py 约 3 步（Ln + 修正 + Mul），gelu_approx 8 步（Mul→Mul→Add→Mul→Tanh→Add→Mul→Mul），约 2.7 倍。新挑战：每步的舍入误差累积（精度预算）、中间值溢出风险（大 x 时 x³）、调试复杂度（需分段定位是哪一步引入误差）。
</details>

答完 4 道选择题后，运行下面的判分 cell（全对再进入第 5 节）。


In [ ]:
# 衔接自查：填入你的答案，运行判分（全对输出"可进入 7.5"）
# 将下面的 'X' 替换为你的选择（A/B/C/D），第5题填关键词
my_answers = {
    1: 'X',  # Flash vs 完整版的本质区别
    2: 'X',  # 节点验收失败后 harness 的处理
    3: 'X',  # "分档读法"指什么
    4: 'X',  # .cannbot/ 存放什么
}
correct = {1: 'B', 2: 'B', 3: 'B', 4: 'B'}
print("衔接自查判分")
print("═" * 35)
score = 0
for q, ans in my_answers.items():
    ok = ans == correct[q]
    score += ok
    print(f"  Q{q}: {'✓' if ok else '✗ 正确答案 ' + correct[q]}")
print("═" * 35)
if score == 4:
    print("✓ 全部通过，可进入 7.5！")
else:
    print(f"得分 {score}/4 → 回看阶段一/二对应章节后重试")

## 5. 精度预算：8 步链的误差会失控吗？

先不进 Lab，第一件事是回答第 3 节的问题 1。xlog1py（基础课程 02.01~02.02，自学：阶段二）计算链 3 步，你看到的完整版实录精度全绿；gelu_approx 的计算链有 **8 步**（Mul→Mul→Add→Mul→Tanh→Add→Mul→Mul），是它的 2.7 倍。直觉上误差会随步数累积，累积多快、会不会突破红线？**先猜，再实验**。本节两件事：① 用实验建立**精度预算**概念；② 指挥完整版交付 gelu_approx，判读你自己产出的验收报告。

### 5.1 从已知的 rtol/atol 说起（回顾）

精度验收判据（基础课程 01.02 · add_custom 实操时用过）：`|impl − golden| ≤ atol + rtol × |golden|`，课程红线 **rtol=1e-5 / atol=1e-7**。

单步舍入有多大？fp32 在 1 附近的 ulp（最小表示间隔）约 **1.19e-7**，和 atol 同一量级。把三个算子的链长与累积误差摆在一起：

| 算子（出处） | 链长 | 累积误差（≈√N·ulp） | 与红线的关系 |
|---|---|---|---|
| add_custom（基础课程 01.02） | 1 步 | ~1 ulp | rtol 项主导，从未担心过精度 |
| xlog1py（基础课程 02.01~02.02） | 3 步 | ~√3 ≈ 1.7 ulp | 仍远小于容差，远小于正在变成小于 |
| **gelu_approx（本节）** | **8 步** | ~√8 ≈ 2.8 ulp | **会不会越线？5.2 的实验回答** |

### 5.2 先猜，再实验

先写下猜测（一会儿对照）：8 步链误差约是单步的 ______ 倍？会超 atol 吗？然后运行实验：用 fp32 逐位模拟 gelu_approx 的 8 步链，与 fp64 golden 对比，统计 50000 个随机样本。实验跑了两个变体：**变体 A** 把 tanh 当作精确（fp64 计算后只舍入一次），只测 8 步链自身的舍入累积；**变体 B** 用 np.tanh（fp32，带函数自身误差），更接近真实实现。


In [ ]:
# 精度预算实验：gelu 8 步链的两种 fp32 模拟 vs fp64 golden（CPU，无 NPU）
import numpy as np

F32 = np.float32

def gelu_chain(x, tanh_impl):
    """8 步链逐位 fp32 模拟；tanh_impl: 'exact'=精确 tanh 仅一次舍入 | 'np32'=numpy fp32"""
    c = (x * x) * x                     # 步1-2: x³
    c = c * F32(0.044715)               # 步3
    inner = x + c                       # 步4
    t = inner * F32(0.7978845608)       # 步5: ×√(2/π)
    if tanh_impl == 'np32':
        th = np.tanh(t).astype(F32)     # numpy fp32 tanh（≈0.5 ulp 函数误差）
    else:
        th = np.tanh(t.astype(np.float64)).astype(F32)  # 精确 tanh，仅一次舍入
    r = x * (th + F32(1.0))             # 步6-7: +1、×x
    return r * F32(0.5)                 # 步8

def golden(x):
    x64 = x.astype(np.float64)
    inner = x64 + 0.044715 * x64**3
    return 0.5 * x64 * (1.0 + np.tanh(inner * 0.7978845608))

rng = np.random.default_rng(42)
x = rng.uniform(-6, 6, 50000).astype(F32)
g = golden(x)
tol = 1e-7 + 1e-5 * np.abs(g)

print("精度预算实验（50000 样本，x ∈ [-6, 6]，atol=1e-7 / rtol=1e-5）")
print("═" * 60)
for name, impl in [("变体A：tanh 精确（只测 8 步舍入累积）", "exact"),
                   ("变体B：np.tanh fp32（叠加函数自身误差）", "np32")]:
    z = gelu_chain(x, impl)
    d = np.abs(z.astype(np.float64) - g)
    bad = d > tol
    print(f"{name}")
    print(f"  最大误差 {np.max(d):.2e}   超差 {np.sum(bad)} 个（{np.sum(bad)/len(bad)*100:.2f}%）")
print("═" * 60)
negband = (x < -3) & (x > -6)   # 负值暴露带：golden 输出 |z| ≈ 1e-7~1e-4
zA = gelu_chain(x, "exact")
dA = np.abs(zA.astype(np.float64) - g)
print(f"变体A 负值暴露带 [-6,-3] 最大误差 {dA[negband].max():.2e}")
print(f"  → 距 atol=1e-7 红线裕量仅 {100*(1-dA[negband].max()/1e-7):.0f}%")
print("→ 解读见 7.5.3：√N 可控 / 预算紧张 / 任何额外误差源都会越线")


### 5.3 实验揭示了什么：精度预算

对照你的猜测，读两个数字：

1. **纯累积可控**：变体 A（tanh 精确）超差 **0 个**，误差按 √N 增长（不是线性），8 步仍在容差内；
2. **但预算紧张，任何额外误差源都会越线**：负值暴露带（x ∈ [-6,-3]）最大误差 ~7.7e-8，距 atol=1e-7 只剩 **~23% 裕量**；变体 B 只叠加了 np.tanh 自身 ~0.5 ulp 的函数误差，超差率立刻从 0% 跳到 2.6%。

<details><summary>📖 选读：为什么是 √N 而不是 N（30 秒版，用掷骰子类比解释）</summary>

掷 8 次骰子求总点数：总偏差不是单次的 8 倍，而是 √8 ≈ 2.8 倍：每次偏差有正有负，相互抵消一部分（随机游走规律）。浮点舍入同理：每步舍入方向近似随机，N 步链总误差 ≈ √N × 单步。7.3 的模拟（3 步→8 步误差比 ≈ 1.6 ≈ √8/√3）验证的正是这条规律。**记住结论 + 知道实验可验证即可，推导过程可跳过。**

</details>

于是有了今天的核心概念：

> **精度预算**：容差允许的总误差，按 √N 规律分摊到每一步。预算紧张时，任何额外误差源（一条精度不够的指令、一次抵消放大）都可能把实现推过红线。

gelu_approx 的需求卡把**必须分段**（|x| > 阈值时直接 z = x 或 z = 0）列为实现红线，正是出于这个预算：饱和区跳过整条计算链，预算消耗归零。至于 NPU 向量指令的精度到底够不够、负值带会发生什么，03.02 节实测时将亲手修复。

### 5.4 v3.0.0 机制速讲（约 15min）

> **视角切换**：基础课程 02.01 已从**决策者视角**看过这套机制的三大设计思想（验收对偶 / 边界静默 / 过程产物分离）与 xlog1py 实录，现在换**指挥官视角**，亲手触发它的每个环节，验证基础课的每一条思想。

> **术语卡**：
>
> | 术语 | 一句话 | 类比 |
> |---|---|---|
> | **harness** | 工作流编排程序，按 YAML 流程定义拉起各角色 | 剧组执行导演 |
> | **verifier** | 逐节点独立验收，不代跑测试 | 质检员 |
> | **精度预算** | 多步计算链中容差允许的总误差（5.2 实验测得） | 每道工序的公差 |

<img src="./images/03.01_workflow_nodes.png" alt="完整版 v3.0.0：三角色 × 七节点工作流全景（交互时间轴 + 自愈回环）" width="700px">

若某节点自测失败，执行者会加载对应调试 Skill 自行修复再自测；harness 层面还有节点级自动重试（默认 3 次，图中回环）。观察点不是它会不会错，而是错了之后系统如何自愈。


## 6. Lab A：gelu_approx 实操（70min）

### 6.1 读需求卡（10min）

打开 `src/requirement_card.md` 通读，它就是基础课程 01.02 需求五要素的完整版形态（公式/dtype/shape/红线逐一固化）。注意：
- **公式**：`z = 0.5 · x · (1 + tanh(√(2/π) · (x + 0.044715x³)))`
- **精度红线**：大 |x| 时中间值 x³ 的精度损失 + Tanh 指令精度
- **实现红线**：必须分段（|x| > 阈值时直接 z = x 或 z = 0）

### 6.2 发起需求与四项确认（10min）

> **预期现象**：PM 先提示“请先不要离开”，然后一次问 ≤5 个选择题：基础课程 01.02 的需求五要素，此刻由 PM 逐项与你确认。

| 待决项 | 建议答复 |
|---|---|
| dtype | 仅 float32 |
| 输入约定 | 等形连续 |
| 测试规模 | 最小集 |
| 架构 | SIMD Vector（MemBase） |

> **预期现象**：答完收到“需求已确认，后续可以静默开发了”，之后 PM 不再提问，转入后台组装工作流。

### 6.3 观察推进（35min）

```bash
tmux has-session -t ops-direct-invoke-XXXX
python3 -c "import json; s=json.load(open('<work_dir>/.workflow/status.json')); print({k:(v['status'],v['retries']) for k,v in s['tasks'].items()})"
```

> **预期现象**：0.0 和 0.1 同时 running（并行）；节点验收失败时 retries +1（自动重试）。

**观察清单**：
- [ ] 0.0 ∥ 0.1 并行启动 + 节点失败后的 harness 自动重试
- [ ] `.workflow/sessions/` 里 executor 与 verifier 是两个独立会话
- [ ] 若出现零产出空转，检查权限配置（external_directory: allow）


In [ ]:
# 工作流推进观察器：读取 status.json，pretty-print 各节点状态
# 实操环境运行前先修改变量 work_dir 为你的实际路径
import json, os, glob

work_dir = None  # ← 填入你的 .cannbot/<任务名>/workflow<N>/ 路径
# 示例: work_dir = "/home/user/ws-advanced/.cannbot/gelu_approx/workflow1"

if work_dir and os.path.exists(os.path.join(work_dir, '.workflow/status.json')):
    s = json.load(open(os.path.join(work_dir, '.workflow/status.json')))
    print("节点状态（实时读取）")
    print("═" * 50)
    for task_id, info in sorted(s['tasks'].items()):
        status, retries = info['status'], info['retries']
        icon = {'pass': '✓', 'running': '→', 'verifying': '◇', 'fail': '✗', 'pending': '·'}.get(status, '?')
        retry_note = f" (重试 ×{retries})" if retries > 0 else ""
        print(f"  [{icon}] {task_id:<6s} {status:<12s}{retry_note}")
    n_pass = sum(1 for v in s['tasks'].values() if v['status'] == 'pass')
    print("═" * 50)
    print(f"  进度: {n_pass}/{len(s['tasks'])} 节点通过")
else:
    # 真机实录 demo（数据出处：xlog1py v3.0.0 完整版工作流真机运行，课程开发实录）
    print("（未找到工作流目录——展示 xlog1py 工作流真机实录状态）")
    print("═" * 50)
    demo = {'0.0': ('pass', 1), '0.1': ('pass', 1), '1': ('pass', 1),
            '2': ('pass', 0), '3': ('pass', 0), '4': ('pass', 0)}
    for tid, (st, rt) in sorted(demo.items()):
        icon = '✓' if st == 'pass' else '→'
        rn = f" (重试 ×{rt})" if rt > 0 else ""
        print(f"  [{icon}] {tid:<6s} {st:<12s}{rn}")
    print("═" * 50)
    print("  6/6 节点通过（3 次重试全部自愈，总耗时 ~3.6h）")
    print("  （出处：xlog1py 完整版真机实录；你的 gelu 运行会产生自己的 status.json）")
    print("\n命令备忘：")
    print("  tmux has-session -t ops-direct-invoke-XXXX")
    print("  tail -f <work_dir>/orchestrator.log")

### 6.4 判读验收报告（10min）

打开任一份 `<节点ID>-验收报告.md`，对照四要素判读：基础课程 02.01 读的是**别人实录里的**验收报告，今天判读的是**自己跑出来的**：

| 要素 | 看什么 | 红旗信号（打回） |
|---|---|---|
| ① 验收对象与证据 | 输入版本、交付件路径、会话 jsonl 齐全且可打开 | 引用不存在的文件 / 版本对不上 |
| ② 逐项结论表 | 每项通过/失败是否附**事实依据** | 无依据的主观结论 |
| ③ 问题与修改意见 | 编号、严重程度、完成判据三件套 | 只写问题不给判据，无法复验 |
| ④ 最终结论 | pass/fail + 对下游节点的影响 | 结论与逐项表矛盾 / 回避下游影响 |

### 6.5 收尾（5min）

```bash
cd tests && python3 -m pytest test_gelu_approx_blackbox.py -q
```

全量测试应转绿。若整轮退出（某节点重试耗尽），体验 PM 的下一轮规划。

## 7. 要点回顾

**本章数字锚点**：8 步计算链 · 23% 预算裕量 · 3 角色 7 节点

- **精度预算**：8 步链误差按 √N 增长、纯累积可控，但负值带裕量仅 ~23%，预算紧张是 gelu 的结构性属性；
- **分段红线**：饱和区跳过计算链，预算消耗归零（需求卡实现红线的由来）；
- **v3.0.0 机制**：三角色 + 七节点 + 节点级自动重试（默认 3 次）；
- **边界静默**：启动前问卷 → 静默执行 → 轮次间决策；
- **验收报告**：放行依据，四要素缺一不可。

**下一站**：[03.02 · 多维度调试](./03.02_operator_debug.ipynb)将**亲手踩**进负值带的精度预算现场，并把三个根因分层挖出来。


## 8. 带走物

| 类别 | 内容 |
|---|---|
| **一句话** | 误差按 √N 累积，预算紧了要分段 |
| **工具** | 精度预算实验 + 完整版 v3.0.0（指挥官视角） |
| **产物** | gelu_approx 完整工程 + 全量测试证据 |
| **模板** | 验收报告四要素判读（对象与证据 / 逐项结论 / 问题清单 / 最终结论） |
| **下一步** | [03.02](./03.02_operator_debug.ipynb)：生成对了，但 4 个用例超差 |

## 9. 章末检查点

1. **（单选）** v3.0.0 中，某节点验收失败后 harness 的处理是？
   - A. 暂停等用户修复
   - B. 自动重试该节点，失败原因注入下一次执行提示词
   - C. 跳过继续
   - D. 立即作废

2. **（单选）** gelu_approx 的 8 步计算链中，精度预算估算的最终误差约？
   - A. √8 · ulp(fp32) ≈ 3.6e-7
   - B. 8 · ulp(fp32) ≈ 4.8e-7
   - C. ulp(fp32) ≈ 6e-8
   - D. 0（浮点运算是精确的）

3. **（简答）** 为什么 gelu_approx 需要分段实现（|x| > 阈值时跳过多项式）？用精度预算的语言解释。

<details><summary>参考答案</summary>

1. **B** 2. **A**（√N 不是 N，因为舍入误差部分相互抵消）
3. 当 |x| > 20 时，x³ 的中间值精度损失累积（加上 0.044715 乘法、加法、tanh 的舍入），总误差超出 rtol=1e-5 的预算。分段跳过多项式直接 z = x（正）或 z = 0（负），因为 tanh 在大参数下已饱和为 ±1，等价于跳过整个计算链。
</details>


In [ ]:
# 产物自查（交付①）：自动核验四件产物
import os, glob, json

repo = None  # ← 填你的工作区路径（如 ~/ws-advanced）
# repo = "/home/user/ws-advanced"

print("产物自查（交付①）")
print("═" * 50)
if repo:
    op = os.path.join(repo, "operators/gelu_approx")
    checks = [
        ("工程四件套", all(os.path.exists(os.path.join(op, f)) for f in
         ["csrc/ops/gelu_approx/op_kernel/gelu_approx_kernel.cpp",
          "csrc/ops/gelu_approx/op_kernel/gelu_approx_launch.h",
          "csrc/ops/gelu_approx/op_plugin/gelu_approx_plugin.cpp",
          "CMakeLists.txt"])),
        ("测试报告 JSON", bool(glob.glob(os.path.join(op, "tests/**/blackbox_report.json"), recursive=True))),
        ("台账 YAML", bool(glob.glob(os.path.join(repo, ".cannbot/**/workflow*.yaml"), recursive=True))),
        ("验收报告", bool(glob.glob(os.path.join(repo, ".cannbot/**/*.md"), recursive=True))),
    ]
    for name, ok in checks:
        print(f"  [{'✓' if ok else '✗'}] {name}")
    print("═" * 50)
    all_ok = all(c[1] for c in checks)
    print(f"  {'✓ 交付① 完成' if all_ok else '✗ 有未完成项，回看 Lab A'}")
else:
    print("  （填入 repo 路径后运行自动核验）")
    print("  需检查：工程四件套 / 测试 JSON / 台账 YAML / 验收报告")

## 10. 产物自查表（交付 ①）

| 项 | 标准 | 自查 |
|---|---|---|
| 工程完整性 | `operators/gelu_approx/` 四件套可构建 | ☐ |
| 测试通过 | 黑盒全量绿（报告 JSON 落盘） | ☐ |
| 台账完整 | `.cannbot/` 含 YAML + 验收报告 + 会话记录 | ☐ |
| 判读能力 | 能解读任一验收报告的证据→结论链 | ☐ |

## 11. 课后练习

> 练习答案：[`answer/03.01_answer.txt`](./answer/03.01_answer.txt)

1. **（单选题）** 8 步链的纯舍入累积误差约为 √8 × ulp（≈3.4× 单步）而非 8 × ulp，原因是？
   - A. 编译器自动消除了中间舍入
   - B. 各步舍入误差正负随机、部分相互抵消（统计规律）
   - C. NPU 向量指令做了补偿设计
   - D. ulp 的定义随链长变化

2. **（单选题）** 双变体实验中，变体 A（tanh 精确）负值带 0 超差、变体 B（np.tanh）超差 2.6%，这组对照说明？
   - A. numpy 的 tanh 实现有 bug
   - B. 8 步链本身必然超差
   - C. 预算只剩 23% 裕量时，任何额外误差源（哪怕 0.5 ulp 的函数误差）都可能越线
   - D. 应该放宽 atol 到 1e-6

3. **（简答）** 需求卡为什么把必须分段（|x| > 阈值时直接 z = x 或 0·x）列为实现红线？请用精度预算的语言回答，并说明为什么阈值不能取 5（提示：|z_true(-5.1)| ≈ 1.02e-7）。

4. **（编程实践·必做）** **gelu 泛化版（项目线）**：把你在 Lab A 交付的 gelu_approx 扩展为 float16/float32 双 dtype。动手前先**重做精度预算**：fp16 在 1 附近的 ulp ≈ 9.77e-4，远大于 fp32 的 1.19e-7：容差标准必须按 dtype 语义重新设定（参照阶段二 01.02 练习 sigmoid 泛化版的三层标准：有限值容差 + 特殊值精确 + NaN 语义）。完成后记录：fp16 的分段阈值是否需要调整、测试用例矩阵如何设计。

5. **（编程实践·选做）** 判读训练：打开任一次完整版运行的节点验收报告，以 verifier 视角写出放行或打回的裁决及三条事实依据（对照 6.4 的四要素模板）。
